In [ ]:
import os
import torch
from bioio import BioImage, Scale
import bioio_tifffile
import numpy as np
import stackview

print (f"Using cuda device: {torch.cuda.is_available()}")

In [ ]:
import vistiq
from vistiq.core import ArrayIteratorConfig
from vistiq.io import ImageLoaderConfig, ImageLoader
from vistiq.preprocess import DoGConfig, DoG
from vistiq.segment.analysis import RegionAnalyzer, RegionAnalyzerConfig
from vistiq.segment.select import RegionFilter, RegionFilterConfig, RangeFilterConfig, RangeFilter
from vistiq.segment.label import MicroSAMSegmenterConfig, MicroSAMSegmenter
from vistiq.segment.postprocess import WatershedConfig, Watershed

# Load image

In [ ]:
path="/standard/vol191/siegristlab/Microsam_Segmentation/Conditional Split/control_24+48/DCP1/1_Dpn.tif"

ilc = ImageLoaderConfig(squeeze=True, rename_channel={"Channel:0:0": "Dpn"}, substack="Z:65-70")
img, metadata = ImageLoader(ilc).run(path)
metadata

# Set embedding path

In [ ]:
z_start = 0
z_end = metadata["dims"].Z
embedding_dir = "/standard/vol191/siegristlab/Sagar/microsam/embeddings/"
embedding_paths = [os.path.join(embedding_dir, ch_label, f"{z_start}-{z_end}") for ch_label in metadata["channel_names"]]

# Preprocess

In [ ]:
dogc = DoGConfig(
    sigma_low=1.0, 
    sigma_high=12.0
)
dimg,_ = DoG(dogc).run(img)

# Define Region Analyzer

In [ ]:
itc = ArrayIteratorConfig(slice_def=(-3,-2,-1))
rac = RegionAnalyzerConfig(
    properties=["volume", "cross_sectional_area", "area", "aspect_ratio", "bbox"], 
    iterator_config=itc, 
    output_type="dataframe"
)
ra = RegionAnalyzer(rac)

# Define Region Filter

In [ ]:
rfc = RegionFilterConfig(filters=[
    RangeFilter(
        RangeFilterConfig(
            attribute="cross_sectional_area", 
            range=(15.0,2000.0)
        )
    ),
    RangeFilter(
        RangeFilterConfig(
            attribute="volume",
            range=(10,20000)
        )
    ),
])
rf = RegionFilter(rfc)

# Segment

In [ ]:
segc = MicroSAMSegmenterConfig(
    embedding_path=embedding_paths[0], 
    do_regions=True, 
    region_analyzer=ra, 
    region_filter=rf
)
dmask, dlabels, dresults = MicroSAMSegmenter(segc).run(dimg, metadata=metadata)

In [ ]:
stackview.slice(dlabels, continuous_update=True)

# Results

In [ ]:
dresults.describe()

In [ ]:
from joblib import Parallel, delayed

def check_labels(labels, results, i):
    coords = np.argwhere(labels == i)
    top_left = results[results.index==i][["bbox-0", "bbox-1", "bbox-2"]].to_numpy().squeeze()
    bottom_right = results[results.index==i][["bbox-3", "bbox-4", "bbox-5"]].to_numpy().squeeze()
    return np.array_equal(top_left, coords.min(axis=0)) and np.array_equal(bottom_right, coords.max(axis=0)+1)

label_nums = np.unique(dlabels)
results = Parallel(n_jobs=-1, verbose=10, batch_size=4)(delayed(check_labels)(dlabels, dresults, i) for i in range(1, label_nums[-1]+1))
assert all(results)

# Postprocess Labels: Watershed

In [ ]:
wsc = WatershedConfig(
    footprint=(3,15,15), 
    h=2.5, 
    min_distance=0.5, 
    compactness=10
)
wlabels = Watershed(wsc).run(dlabels, metadata)

In [ ]:
import stackview
stackview.slice(np.concatenate([dlabels, wlabels], axis=-1), continuous_update=True)